# Pitch Homography

Per-game step: map image pixels to pitch coordinates (metres on a 105×68m FIFA-standard pitch).

## Model: PnLCalib (HRNetV2-W48)
Pretrained encoder-decoder that detects pitch keypoints and lines, then estimates a full 3×4 camera projection matrix.

## Post-processing layers
1. **Non-gameplay filter** — skip replays/close-ups via scoreboard edge density
2. **Player sanity check** — reject projections where <50% of players land within the pitch bounds
3. **Line alignment check** — verify projected lines overlap with bright pixels in the image
4. **Temporal consistency** — reject single-frame outliers; require 3 consecutive frames before accepting a large shift
5. **EMA smoothing** — blend consecutive projections to eliminate jitter

## Coverage across 16 matches (v2, 60s clips at 10:00 into first half)
| Tier | Matches | Coverage |
|---|---|---|
| **Tier 1 (>80%)** | jez-ars, dec-mla, jez-jed, sut-pet, sut-mla, mla-bud-2, jed-ars | 84–100% |
| **Tier 2 (60–80%)** | pet-mor, pet-bok, bok-jed, bud-sut | 64–75% |
| **Tier 3 (<60%)** | mor-bud, mor-ars, ars-dec | 10–43% |
| **Crash** | bok-jed-2, mla-bud | — |

In [ ]:
import sys
import json
import subprocess
import cv2
import numpy as np
import matplotlib.pyplot as plt
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.config
importlib.reload(src.config)
from src.config import Config

# Period boundaries for all matches
periods_data = json.loads((PROJECT_ROOT / 'data' / 'period_detection_results.json').read_text())
period_map   = {r['slug']: r for r in periods_data if r['status'] == 'ok'}

print(f'Loaded. {len(period_map)} matches with period data.')

## Run PnLCalib on a Match

Outputs an annotated MP4 to `output/{slug}_pnlcalib_{duration}s.mp4`.

In [ ]:
GAME_SLUG    = 'bud-sut'  # <- change this per game
DURATION_SEC = 60          # clip duration in seconds
ALPHA        = 0.3         # EMA smoothing (0.1 = very smooth, 0.5 = responsive)

p        = period_map[GAME_SLUG]
fh_start = p['first_half_start_frame']
fps      = p['fps']
# OFFSET_MIN is derived from fh_start so PnLCalib receives the actual game start
offset_min = fh_start / fps / 60

print(f'Game: {GAME_SLUG}')
print(f'First half starts at frame {fh_start} ({offset_min:.2f} min into broadcast)')
print(f'Clip: {DURATION_SEC}s from kickoff')

In [ ]:
result = subprocess.run(
    [
        sys.executable, '-m', 'src.run_pnlcalib_video',
        '--match',        GAME_SLUG,
        '--offset_min',   str(offset_min),
        '--duration_sec', str(DURATION_SEC),
        '--alpha',        str(ALPHA),
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Results — All Matches

5 sampled frames from each match's clip, sorted by coverage.

In [ ]:
# Coverage from the v2 run (60s clips at kickoff).
# Update these numbers after re-running with new settings.
COVERAGE = {
    'jez-ars':  100.0, 'dec-mla':   97.6, 'jez-jed':  95.9, 'sut-pet':  91.7,
    'sut-mla':   91.5, 'mla-bud-2': 84.1, 'jed-ars':  82.5, 'pet-mor':  74.9,
    'pet-bok':   74.0, 'bok-jed':   66.1, 'bud-sut':  64.5, 'mor-bud':  43.0,
    'mor-ars':   21.3, 'ars-dec':   10.5,
}

for slug in sorted(COVERAGE, key=COVERAGE.get, reverse=True):
    path = Config.OUTPUT_HOMOGRAPHY_DIR / f'{slug}_pnlcalib_{DURATION_SEC}s.mp4'
    if not path.exists():
        print(f'MISSING: {slug}')
        continue

    cap   = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fig, axes = plt.subplots(1, 5, figsize=(24, 4), facecolor='white')
    for j, pct in enumerate([0.1, 0.3, 0.5, 0.7, 0.9]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(total * pct))
        ret, frame = cap.read()
        if ret:
            axes[j].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[j].axis('off')
    cap.release()

    coverage = COVERAGE[slug]
    color    = 'green' if coverage >= 80 else ('orange' if coverage >= 60 else 'red')
    fig.suptitle(f'{slug.upper().replace("-", " vs ")} — {coverage:.1f}% coverage',
                 fontsize=14, fontweight='bold', color=color, y=1.02)
    plt.tight_layout()
    plt.show()
    print()

## Fallback: Optical Flow + Manual Keypoint Seeds

When PnLCalib fails on a frame, the pipeline now tries two fallbacks before reusing the stale projection:

1. **Optical-flow propagation** (`src/camera_motion.py`) — LK flow on pitch features re-estimates the plane homography from the last trusted frame. Handles short PnLCalib dropouts inside a stable shot.
2. **Manual keypoint seeds** (`src/manual_calibration.py`) — you click 4+ named pitch landmarks on a handful of reference frames per Tier-3 match. The resulting homographies are stored under `data/manual_calibration/` and used to seed the flow tracker on long cold starts.

Both are wired into `src.run_pnlcalib_video` automatically. No CLI changes needed — disable with `--no_flow` / `--no_manual_seeds` if you want to benchmark against the old behaviour.

### 1. Pick a reference frame

Scrub to a moment where pitch lines are clearly visible (kickoff, set pieces, wide shots). Lower = easier. Record the frame number — you'll label landmarks on it below.

In [ ]:
# Preview frames at evenly spaced offsets from kickoff so you can pick a good one.
from src.manual_calibration import grab_frame

GAME_SLUG = 'ars-dec'          # Tier 3 match to work on
OFFSETS_SEC = [30, 120, 300, 600, 900, 1500]   # seconds into first half

p = period_map[GAME_SLUG]
fh_start, fps = p['first_half_start_frame'], p['fps']

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, offset_s in zip(axes.flat, OFFSETS_SEC):
    fn = fh_start + int(offset_s * fps)
    try:
        frame = grab_frame(GAME_SLUG, fn)
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'frame {fn} (+{offset_s}s)')
    except Exception as e:
        ax.set_title(f'err {fn}: {e}')
    ax.axis('off')
plt.tight_layout(); plt.show()

### 2. Label landmarks on the chosen frame

`%matplotlib widget` is required — click on the video frame to record the currently-selected landmark's image position. The dropdown auto-advances after each click. 4+ points is the minimum, 8-12 well-distributed points gives a noticeably better H. Save writes to `data/manual_calibration/{slug}_frame_{N:07d}.json`.

In [ ]:
%matplotlib widget
import importlib, src.manual_calibration
importlib.reload(src.manual_calibration)
from src.manual_calibration import grab_frame, build_labeling_widget, calibration_path, load_calibration

FRAME_NUMBER = fh_start + int(300 * fps)   # edit to the offset you picked above

frame = grab_frame(GAME_SLUG, FRAME_NUMBER)

# Resume previous annotation if we've saved this frame before
existing_path = calibration_path(GAME_SLUG, FRAME_NUMBER)
existing = load_calibration(existing_path) if existing_path.exists() else None

build_labeling_widget(GAME_SLUG, frame, FRAME_NUMBER, existing=existing)

### 3. Preview the homography

Draws the projected pitch lines using the manually-fitted H (z=0 lines only — goal posts are skipped for plane homographies). If lines don't hug white paint, add more / better-distributed points and re-save.

In [ ]:
%matplotlib inline
from src.manual_calibration import load_calibration, calibration_path
from src.run_pnlcalib_video import project_lines

cal = load_calibration(calibration_path(GAME_SLUG, FRAME_NUMBER))
P = np.array(cal.P_pnlcalib_convention)

frame = grab_frame(GAME_SLUG, FRAME_NUMBER)
frame = project_lines(frame.copy(), P, color=(0, 0, 255), thickness=2)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f'{GAME_SLUG}  frame {FRAME_NUMBER}  |  reproj err = {cal.reprojection_error_m:.2f} m  |  {len(cal.correspondences)} pts')
plt.axis('off'); plt.show()

### 4. List saved seeds for this match

The runner picks these up automatically — no further wiring needed.

In [ ]:
from src.manual_calibration import load_match_seeds
seeds = load_match_seeds(GAME_SLUG)
print(f'{len(seeds)} seed(s) for {GAME_SLUG}')
for fn in sorted(seeds):
    print(f'  frame {fn}')